In [ ]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
import pegasus as pg
from gene_lists import *

# Set variables
resolutions = [0.3] # This takes a while per res

# Load data

In [ ]:
data = pg.read_input(plate_path)
data.obs['sample'] = data.obs['sample'].str.replace('sample_', '')
data = data[~data.obs['sample'].str.endswith(tuple(['WGE', 'Hipp', 'Thal']))]
data.obs['sublibrary'] = [x[1] for x in data.obs.index.str.split('__s')] 
data

In [ ]:
data.X

In [ ]:
data.obs.head()

In [ ]:
data.obs['sample'].value_counts()

In [ ]:
data.obs['sublibrary'].value_counts()

# Preprocessing

In [ ]:
pg.qc_metrics(data, min_genes=500, max_genes=6000, mito_prefix='MT-', percent_mito=5)

In [ ]:
df_qc = pg.get_filter_stats(data)
df_qc

In [ ]:
pg.qcviolin(data, plot_type='gene', dpi=100)

In [ ]:
pg.filter_data(data)

In [ ]:
pg.identify_robust_genes(data)

In [ ]:
pg.log_norm(data)

In [ ]:
# For the downstream analysis, we may need to make a copy of the count matrix, in case of coming back to this step and redo the analysis
data_trial = data.copy()

In [ ]:
pg.highly_variable_features(data_trial)

In [ ]:
data_trial.var.loc[data_trial.var['highly_variable_features']].sort_values(by='hvf_rank')

In [ ]:
pg.hvfplot(data_trial, dpi=200)

In [ ]:
pg.pca(data_trial)

In [ ]:
pg.neighbors(data_trial)

In [ ]:
pg.louvain(data_trial)

In [ ]:
data_trial.obs['louvain_labels'].value_counts()

In [ ]:
pg.compo_plot(data_trial, 'sample', 'louvain_labels')

# Visualization

In [ ]:
pg.tsne(data_trial)

pg.umap(data_trial)

In [ ]:
pg.scatter(data_trial, attrs=['louvain_labels', 'sublibrary'], basis='tsne')

In [ ]:
pg.scatter(data_trial, attrs=['louvain_labels', 'sublibrary'], basis='umap')